# SkillScope ID - Fine-tuning NER Skill Extraction

Notebook ini dibuat untuk dijalankan di Kaggle. Tujuannya adalah melatih model Named Entity Recognition untuk mengekstrak `HSkill`, `SSkill`, dan `Tech` dari lowongan kerja Indonesia, membandingkan IndoBERT dengan XLM-RoBERTa, mengevaluasi performa secara lengkap, lalu mengekspor model terbaik ke ONNX untuk web demo HTML/CSS/JS.

Catatan penting: notebook ini tidak menjanjikan akurasi tinggi secara absolut. Performa tinggi dikejar melalui split data yang benar, monitoring validation metrics, early stopping, checkpoint terbaik, evaluasi test set, error analysis, dan validasi ONNX.

## 0. Setup Kaggle

Cell ini menyiapkan dependency yang dibutuhkan. Di Kaggle sebagian package biasanya sudah tersedia, tetapi install eksplisit membuat notebook lebih reproducible. Jika dependency sudah lengkap, cell ini tetap aman dijalankan.

In [ ]:
!pip -q install transformers datasets evaluate seqeval accelerate scikit-learn pandas numpy matplotlib seaborn onnx onnxruntime "optimum[onnxruntime]"

## 1. Import Library

Cell ini memuat library untuk parsing dataset, EDA, training HuggingFace Trainer, evaluasi NER, visualisasi, dan export ONNX.

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import shutil
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset, DatasetDict
from IPython.display import display
from onnxruntime.quantization import QuantType, quantize_dynamic
from optimum.onnxruntime import ORTModelForTokenClassification
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

sns.set_theme(style="whitegrid", context="notebook")

## 2. Konfigurasi Eksperimen

Ubah `DATA_PATH` sesuai lokasi dataset NERSkill.Id di Kaggle. Output eksperimen disimpan ke `/kaggle/working/outputs` agar mudah di-download setelah notebook selesai.

In [ ]:
SEED = 42
DATA_PATH = Path("/kaggle/input/datasets/maidilfitrah/nerskill-id")
OUTPUT_DIR = Path("/kaggle/working/outputs")
PLOTS_DIR = OUTPUT_DIR / "plots"
REPORTS_DIR = OUTPUT_DIR / "reports"
MODELS_DIR = OUTPUT_DIR / "models"
ONNX_DIR = OUTPUT_DIR / "onnx"
WEB_MODEL_DIR = OUTPUT_DIR / "web_model"

for directory in [OUTPUT_DIR, PLOTS_DIR, REPORTS_DIR, MODELS_DIR, ONNX_DIR, WEB_MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_CONFIGS = {
    "indobert": "indobenchmark/indobert-base-p2",
    "xlm_roberta": "xlm-roberta-base",
}

LABEL_LIST = ["O", "B-HSkill", "I-HSkill", "B-SSkill", "I-SSkill", "B-Tech", "I-Tech"]
LABEL2ID = {label: idx for idx, label in enumerate(LABEL_LIST)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}
ENTITY_TYPES = ["HSkill", "SSkill", "Tech"]

MAX_LENGTH = 128
TRAIN_SIZE = 0.80
VALIDATION_SIZE = 0.10
TEST_SIZE = 0.10

TRAINING_CONFIG = {
    "learning_rate": 2e-5,
    "num_train_epochs": 10,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "max_grad_norm": 1.0,
    "early_stopping_patience": 3,
    "save_total_limit": 2,
}

def check_cuda_runtime() -> tuple[bool, str]:
    """Detect CUDA/PyTorch compatibility before a long training run starts."""
    if not torch.cuda.is_available():
        return False, "CUDA tidak tersedia. Aktifkan GPU di Kaggle Settings -> Accelerator."

    try:
        device = torch.device("cuda")
        probe = torch.tensor([1, 2, 3], device=device)
        _ = (probe + 1).detach().cpu().numpy()
        torch.cuda.synchronize()
        return True, "CUDA siap digunakan."
    except Exception as exc:
        return False, f"CUDA terdeteksi tetapi gagal menjalankan kernel PyTorch: {exc}"


CUDA_OK, CUDA_MESSAGE = check_cuda_runtime()
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
CUDA_VERSION = torch.version.cuda
USE_FP16 = CUDA_OK and any(name in GPU_NAME.upper() for name in ["T4", "V100", "A100", "L4"])

print("Torch version:", torch.__version__)
print("Torch CUDA build:", CUDA_VERSION)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", GPU_NAME)
print("CUDA status:", CUDA_MESSAGE)
print("Use fp16:", USE_FP16)

if not CUDA_OK:
    raise RuntimeError(
        CUDA_MESSAGE
        + "\nSolusi Kaggle: restart session, pilih Accelerator GPU T4 jika tersedia, "
        + "atau jalankan ulang setelah mengganti environment. Error 'no kernel image' biasanya berarti build PyTorch/CUDA tidak kompatibel dengan GPU yang didapat."
    )

## 3. Reproducibility

Seed dibuat tetap agar split dan eksperimen lebih mudah direproduksi. Deterministic mode penuh tidak dipaksa karena bisa memperlambat training GPU, tetapi seed dasar tetap dikunci.

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(SEED)

experiment_config = {
    "seed": SEED,
    "data_path": str(DATA_PATH),
    "models": MODEL_CONFIGS,
    "label_list": LABEL_LIST,
    "max_length": MAX_LENGTH,
    "training_config": TRAINING_CONFIG,
}
(OUTPUT_DIR / "experiment_config.json").write_text(json.dumps(experiment_config, indent=2), encoding="utf-8")

## 4. Load Dataset CoNLL/BIO

Parser dibuat fleksibel untuk dua format umum: `token tag` dan `sentence_id token tag`. Header otomatis dilewati jika terdeteksi. Validasi label dilakukan sejak awal agar error dataset tidak masuk diam-diam ke training.

In [ ]:
def resolve_dataset_file(path: Path) -> Path:
    """Return a dataset file path even when DATA_PATH points to a Kaggle input directory."""
    if not path.exists():
        raise FileNotFoundError(f"Dataset tidak ditemukan: {path}. Ubah DATA_PATH sesuai path Kaggle dataset.")

    if path.is_file():
        return path

    candidates = []
    allowed_suffixes = {".txt", ".tsv", ".csv", ".conll"}
    for candidate in path.rglob("*"):
        if candidate.is_file() and candidate.suffix.lower() in allowed_suffixes:
            candidates.append(candidate)

    if not candidates:
        visible_files = [str(item) for item in path.rglob("*") if item.is_file()][:20]
        raise FileNotFoundError(
            "DATA_PATH mengarah ke folder, tetapi tidak ditemukan file dataset "
            f"dengan ekstensi {sorted(allowed_suffixes)} di {path}. "
            f"File yang terlihat: {visible_files}"
        )

    preferred = sorted(
        candidates,
        key=lambda item: (
            "nerskill" not in item.name.lower(),
            item.suffix.lower() not in {".txt", ".tsv"},
            len(str(item)),
        ),
    )[0]
    print(f"DATA_PATH adalah folder. Menggunakan file dataset: {preferred}")
    return preferred


def parse_conll(path: Path) -> list[dict[str, list[str]]]:
    path = resolve_dataset_file(path)

    records = []
    tokens, tags = [], []
    skipped_header = False
    previous_sentence_id = None

    def flush_sentence():
        nonlocal tokens, tags
        if tokens:
            records.append({"tokens": tokens, "ner_tags": tags})
            tokens, tags = [], []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            line = raw_line.strip()
            if not line:
                flush_sentence()
                previous_sentence_id = None
                continue

            parts = line.split("\t") if "\t" in line else line.split()
            lowered = [part.lower() for part in parts]
            if not skipped_header and any(name in lowered for name in ["word", "token", "tag"]):
                skipped_header = True
                continue

            if len(parts) < 2:
                raise ValueError(f"Baris tidak valid di line {line_number}: {raw_line!r}")

            tag = parts[-1].strip()
            if len(parts) >= 4 and parts[0].lower().startswith("sentence"):
                sentence_id = f"{parts[0]} {parts[1]}"
            elif len(parts) >= 3:
                sentence_id = parts[0].strip()
            else:
                sentence_id = None
            token = parts[-2].strip() if len(parts) >= 3 else parts[0].strip()

            if tag not in LABEL2ID:
                raise ValueError(f"Label tidak dikenal {tag!r} di line {line_number}")

            if sentence_id is not None and previous_sentence_id is not None and sentence_id != previous_sentence_id:
                flush_sentence()
            if sentence_id is not None:
                previous_sentence_id = sentence_id

            if token:
                tokens.append(token)
                tags.append(tag)

    flush_sentence()

    if not records:
        raise ValueError("Dataset kosong setelah parsing.")
    if len(records) < 3:
        raise ValueError(
            f"Dataset hanya terbaca sebagai {len(records)} kalimat. "
            "Kemungkinan format kolom belum sesuai. Cek apakah dataset memakai kolom Sentence/Word/Tag, "
            "dan pastikan parser memisahkan kalimat berdasarkan sentence id atau baris kosong."
        )

    return records


records = parse_conll(DATA_PATH)
total_tokens = sum(len(row["tokens"]) for row in records)
print(f"Total sentences: {len(records):,}")
print(f"Total tokens: {total_tokens:,}")
display(pd.DataFrame(records[:3]))

## 5. Helper EDA Entitas BIO

Cell ini mengubah urutan token-label menjadi span entitas utuh. Hasilnya dipakai untuk analisis distribusi entity, panjang span, frasa paling sering, dan error analysis.

In [ ]:
def bio_to_entities(tokens: list[str], tags: list[str]) -> list[dict[str, object]]:
    entities = []
    current = None

    for idx, (token, tag) in enumerate(zip(tokens, tags)):
        if tag == "O":
            if current:
                entities.append(current)
                current = None
            continue

        prefix, entity_type = tag.split("-", 1)
        should_start = prefix == "B" or current is None or current["type"] != entity_type

        if should_start:
            if current:
                entities.append(current)
            current = {"type": entity_type, "tokens": [token], "start": idx, "end": idx}
        else:
            current["tokens"].append(token)
            current["end"] = idx

    if current:
        entities.append(current)

    for entity in entities:
        entity["text"] = " ".join(entity["tokens"])
        entity["length"] = len(entity["tokens"])
    return entities


entity_rows = []
for sentence_id, row in enumerate(records):
    for entity in bio_to_entities(row["tokens"], row["ner_tags"]):
        entity_rows.append({"sentence_id": sentence_id, **entity})

entities_df = pd.DataFrame(entity_rows)
display(entities_df.head(10))
print(f"Total entities: {len(entities_df):,}")

## 6. EDA Ringkasan Dataset

Cell ini membuat statistik dasar yang akan dipakai pada laporan: jumlah kalimat, jumlah token, jumlah entitas, rata-rata panjang kalimat, dan distribusi label BIO.

In [ ]:
sentence_lengths = pd.Series([len(row["tokens"]) for row in records], name="sentence_length")
flat_tags = [tag for row in records for tag in row["ner_tags"]]
label_counts = pd.Series(flat_tags).value_counts().reindex(LABEL_LIST, fill_value=0)
entity_counts = entities_df["type"].value_counts().reindex(ENTITY_TYPES, fill_value=0) if not entities_df.empty else pd.Series(0, index=ENTITY_TYPES)

summary = {
    "total_sentences": int(len(records)),
    "total_tokens": int(total_tokens),
    "total_entities": int(len(entities_df)),
    "avg_sentence_length": float(sentence_lengths.mean()),
    "median_sentence_length": float(sentence_lengths.median()),
    "max_sentence_length": int(sentence_lengths.max()),
    "label_counts": {label: int(count) for label, count in label_counts.items()},
    "entity_counts": {label: int(count) for label, count in entity_counts.items()},
}

(OUTPUT_DIR / "dataset_stats.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(pd.DataFrame([summary]).drop(columns=["label_counts", "entity_counts"]))
display(label_counts.to_frame("token_count"))
display(entity_counts.to_frame("entity_count"))

## 7. Visualisasi EDA: Label dan Entity

Visualisasi ini membantu menjelaskan apakah dataset imbalanced. Jika label `O` jauh dominan, F1 entity-level lebih penting daripada token accuracy karena token accuracy bisa terlihat tinggi walaupun model melewatkan entitas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(x=label_counts.index, y=label_counts.values, ax=axes[0], palette="viridis")
axes[0].set_title("BIO Label Distribution")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Token Count")
axes[0].tick_params(axis="x", rotation=35)

sns.barplot(x=entity_counts.index, y=entity_counts.values, ax=axes[1], palette="crest")
axes[1].set_title("Entity Type Distribution")
axes[1].set_xlabel("Entity Type")
axes[1].set_ylabel("Entity Count")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "label_and_entity_distribution.png", dpi=180)
plt.show()

## 8. Visualisasi EDA: Panjang Kalimat dan Span Entitas

Analisis panjang kalimat digunakan untuk menentukan `MAX_LENGTH`. Jika banyak kalimat melewati 128 token, naikkan `MAX_LENGTH` ke 256 agar konteks tidak banyak terpotong.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(sentence_lengths, bins=40, kde=True, ax=axes[0], color="#2563eb")
axes[0].axvline(MAX_LENGTH, color="red", linestyle="--", label=f"MAX_LENGTH={MAX_LENGTH}")
axes[0].set_title("Sentence Length Distribution")
axes[0].set_xlabel("Tokens per Sentence")
axes[0].legend()

sns.boxplot(x=sentence_lengths, ax=axes[1], color="#1f8a5f")
axes[1].set_title("Sentence Length Boxplot")
axes[1].set_xlabel("Tokens per Sentence")

if not entities_df.empty:
    sns.histplot(data=entities_df, x="length", hue="type", multiple="dodge", discrete=True, ax=axes[2])
axes[2].set_title("Entity Span Length")
axes[2].set_xlabel("Tokens per Entity")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "length_distributions.png", dpi=180)
plt.show()

truncation_rate = float((sentence_lengths > MAX_LENGTH).mean())
print(f"Sentences longer than MAX_LENGTH={MAX_LENGTH}: {truncation_rate:.2%}")

## 9. Visualisasi EDA: Top Entity Phrases

Cell ini menampilkan frasa entitas yang paling sering muncul. Ini berguna untuk memahami vocabulary domain dan menyiapkan contoh demo presentasi.

In [ ]:
if not entities_df.empty:
    top_entities = (
        entities_df.assign(text_norm=entities_df["text"].str.lower())
        .groupby(["type", "text_norm"])
        .size()
        .reset_index(name="count")
        .sort_values(["type", "count"], ascending=[True, False])
    )
    display(top_entities.groupby("type").head(10))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, entity_type in zip(axes, ENTITY_TYPES):
        subset = top_entities[top_entities["type"] == entity_type].head(10)
        sns.barplot(data=subset, y="text_norm", x="count", ax=ax, color="#1f8a5f")
        ax.set_title(f"Top {entity_type}")
        ax.set_xlabel("Count")
        ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "top_entity_phrases.png", dpi=180)
    plt.show()
else:
    print("Tidak ada entitas untuk divisualisasikan.")

## 10. Split Train/Validation/Test

Split dilakukan pada level kalimat, bukan token, agar konteks satu kalimat tidak bocor ke split lain. Test set tidak boleh dipakai untuk tuning; hanya untuk evaluasi final.

In [ ]:
encoded_records = [
    {"tokens": row["tokens"], "ner_tags": [LABEL2ID[tag] for tag in row["ner_tags"]]}
    for row in records
]

train_records, temp_records = train_test_split(encoded_records, test_size=0.20, random_state=SEED, shuffle=True)
validation_records, test_records = train_test_split(temp_records, test_size=0.50, random_state=SEED, shuffle=True)

raw_dataset = DatasetDict({
    "train": Dataset.from_list(train_records),
    "validation": Dataset.from_list(validation_records),
    "test": Dataset.from_list(test_records),
})

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "sentences": [len(train_records), len(validation_records), len(test_records)],
    "tokens": [sum(len(row["tokens"]) for row in split) for split in [train_records, validation_records, test_records]],
})
split_summary["sentence_pct"] = split_summary["sentences"] / split_summary["sentences"].sum()
display(split_summary)
raw_dataset

## 11. Visualisasi Distribusi Label per Split

Cell ini memastikan distribusi label di train, validation, dan test tidak terlalu berbeda. Perbedaan ekstrem dapat membuat evaluasi tidak stabil.

In [ ]:
def split_label_counts(split_records: list[dict]) -> pd.Series:
    labels = [ID2LABEL[tag_id] for row in split_records for tag_id in row["ner_tags"]]
    return pd.Series(labels).value_counts().reindex(LABEL_LIST, fill_value=0)


split_label_df = pd.DataFrame({
    "train": split_label_counts(train_records),
    "validation": split_label_counts(validation_records),
    "test": split_label_counts(test_records),
}).T
display(split_label_df)

plt.figure(figsize=(13, 5))
split_label_df.plot(kind="bar", stacked=True, figsize=(13, 5), colormap="viridis")
plt.title("BIO Label Distribution per Split")
plt.xlabel("Split")
plt.ylabel("Token Count")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "split_label_distribution.png", dpi=180)
plt.show()

## 12. Tokenization dan Label Alignment

Transformer memakai subword tokenization. Label BIO hanya diberikan ke subword pertama; special token dan subword lanjutan diberi `-100` agar diabaikan oleh loss function. Ini penting agar training tidak rusak.

In [ ]:
def tokenize_and_align_dataset(dataset: DatasetDict, model_name: str, max_length: int = MAX_LENGTH):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_and_align_labels(examples):
        tokenized = tokenizer(
            examples["tokens"],
            truncation=True,
            is_split_into_words=True,
            max_length=max_length,
        )

        labels = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)
            aligned = []
            previous_word_id = None
            for word_id in word_ids:
                if word_id is None:
                    aligned.append(-100)
                elif word_id != previous_word_id:
                    aligned.append(word_labels[word_id])
                else:
                    aligned.append(-100)
                previous_word_id = word_id
            labels.append(aligned)

        tokenized["labels"] = labels
        return tokenized

    tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)
    return tokenizer, tokenized_dataset

## 13. Sanity Check Token Alignment

Cell ini wajib dicek sebelum training. Jika label tidak sejajar dengan token, metrik bisa terlihat buruk atau tidak valid.

In [ ]:
debug_tokenizer, debug_tokenized = tokenize_and_align_dataset(raw_dataset, MODEL_CONFIGS["indobert"], MAX_LENGTH)
sample = debug_tokenized["train"][0]
tokens = debug_tokenizer.convert_ids_to_tokens(sample["input_ids"])
aligned_labels = [ID2LABEL[label] if label != -100 else "IGN" for label in sample["labels"]]
alignment_preview = pd.DataFrame({"token": tokens, "label": aligned_labels, "attention_mask": sample["attention_mask"]})
display(alignment_preview.head(40))

## 14. Metric Function

Metrik utama adalah entity-level precision, recall, dan F1 dari `seqeval`. Token accuracy tidak dijadikan metrik utama karena label `O` biasanya sangat dominan.

In [ ]:
def decode_predictions(predictions, labels):
    prediction_ids = np.argmax(predictions, axis=2)
    true_labels = [
        [ID2LABEL[int(label_id)] for pred_id, label_id in zip(prediction, label) if label_id != -100]
        for prediction, label in zip(prediction_ids, labels)
    ]
    true_predictions = [
        [ID2LABEL[int(pred_id)] for pred_id, label_id in zip(prediction, label) if label_id != -100]
        for prediction, label in zip(prediction_ids, labels)
    ]
    return true_predictions, true_labels


def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    true_predictions, true_labels = decode_predictions(predictions, labels)
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }


def make_classification_report(predictions, labels) -> str:
    true_predictions, true_labels = decode_predictions(predictions, labels)
    return classification_report(true_labels, true_predictions, digits=4)

## 15. Training Utilities

Fungsi training memakai best practice untuk mencegah underfitting/overfitting: validation per epoch, early stopping, warmup, weight decay, gradient clipping, mixed precision jika GPU tersedia, dan `load_best_model_at_end=True`.

In [ ]:
def get_training_args(run_name: str) -> TrainingArguments:
    run_dir = MODELS_DIR / run_name
    return TrainingArguments(
        output_dir=str(run_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=TRAINING_CONFIG["learning_rate"],
        per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
        per_device_eval_batch_size=TRAINING_CONFIG["per_device_eval_batch_size"],
        num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
        weight_decay=TRAINING_CONFIG["weight_decay"],
        warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
        max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=TRAINING_CONFIG["save_total_limit"],
        report_to="none",
        fp16=USE_FP16,
        seed=SEED,
        data_seed=SEED,
    )


def trainer_log_history_to_df(trainer: Trainer) -> pd.DataFrame:
    rows = []
    for item in trainer.state.log_history:
        row = dict(item)
        if "epoch" in row:
            rows.append(row)
    return pd.DataFrame(rows)


def plot_training_history(history_df: pd.DataFrame, run_name: str) -> None:
    if history_df.empty:
        print(f"Tidak ada history untuk {run_name}")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    loss_cols = [col for col in ["loss", "eval_loss"] if col in history_df.columns]
    for col in loss_cols:
        sns.lineplot(data=history_df.dropna(subset=[col]), x="epoch", y=col, marker="o", ax=axes[0], label=col)
    axes[0].set_title(f"{run_name} Loss Curve")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    metric_cols = [col for col in ["eval_precision", "eval_recall", "eval_f1"] if col in history_df.columns]
    for col in metric_cols:
        sns.lineplot(data=history_df.dropna(subset=[col]), x="epoch", y=col, marker="o", ax=axes[1], label=col)
    axes[1].set_title(f"{run_name} Validation Metrics")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"training_history_{run_name}.png", dpi=180)
    plt.show()

## 16. Fungsi Training Satu Model

Cell ini menjalankan satu model dari tokenization sampai test evaluation. Hasil utama yang disimpan: best checkpoint, classification report, training history, dan metrik test.

In [ ]:
def train_one_model(run_name: str, model_name: str) -> dict[str, object]:
    print(f"\n=== Training {run_name}: {model_name} ===")
    set_seed(SEED)
    tokenizer, tokenized_dataset = tokenize_and_align_dataset(raw_dataset, model_name, MAX_LENGTH)
    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(LABEL_LIST),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    trainer = Trainer(
        model=model,
        args=get_training_args(run_name),
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=TRAINING_CONFIG["early_stopping_patience"])],
    )

    started_at = time.time()
    train_result = trainer.train()
    train_time_seconds = time.time() - started_at

    history_df = trainer_log_history_to_df(trainer)
    history_df.to_csv(REPORTS_DIR / f"training_history_{run_name}.csv", index=False)
    plot_training_history(history_df, run_name)

    validation_metrics = trainer.evaluate(tokenized_dataset["validation"])
    test_metrics = trainer.evaluate(tokenized_dataset["test"])
    predictions, labels, _ = trainer.predict(tokenized_dataset["test"])
    report = make_classification_report(predictions, labels)

    best_dir = MODELS_DIR / f"{run_name}-best"
    best_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)

    (REPORTS_DIR / f"classification_report_{run_name}.txt").write_text(report, encoding="utf-8")

    metrics = {
        "run_name": run_name,
        "model_name": model_name,
        "best_model_dir": str(best_dir),
        "train_time_seconds": float(train_time_seconds),
        "train_loss": float(train_result.training_loss),
        **{f"validation_{k.replace('eval_', '')}": float(v) for k, v in validation_metrics.items() if isinstance(v, (int, float))},
        **{f"test_{k.replace('eval_', '')}": float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))},
    }
    (REPORTS_DIR / f"metrics_{run_name}.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

    print(report)
    return metrics

## 17. Fine-tuning Model

Cell ini melatih dua model: IndoBERT dan XLM-RoBERTa. Jika waktu Kaggle terbatas, jalankan IndoBERT dulu, simpan output, lalu lanjutkan XLM-RoBERTa di run berikutnya. Untuk laporan, idealnya kedua model selesai agar perbandingan valid.

In [ ]:
all_metrics = []

for run_name, model_name in MODEL_CONFIGS.items():
    metrics = train_one_model(run_name, model_name)
    all_metrics.append(metrics)

comparison_df = pd.DataFrame(all_metrics)
comparison_df.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
(REPORTS_DIR / "model_comparison.json").write_text(comparison_df.to_json(orient="records", indent=2), encoding="utf-8")
display(comparison_df)

## 18. Visualisasi Perbandingan Model

Cell ini membuat grafik perbandingan precision, recall, dan F1 pada test set. Grafik ini bisa langsung dimasukkan ke laporan atau slide.

In [ ]:
metric_cols = ["test_precision", "test_recall", "test_f1"]
plot_df = comparison_df[["run_name", *metric_cols]].melt(id_vars="run_name", var_name="metric", value_name="score")

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="run_name", y="score", hue="metric", palette="crest")
plt.title("Model Comparison on Test Set")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "model_comparison.png", dpi=180)
plt.show()

display(comparison_df.sort_values("test_f1", ascending=False))

## 19. Cek Underfitting dan Overfitting

Cell ini membaca training history dan memberi flag sederhana. Keputusan akhir tetap perlu dilihat dari grafik, tetapi flag ini membantu membuat pembahasan laporan lebih sistematis.

In [ ]:
def diagnose_training(run_name: str) -> dict[str, object]:
    history_path = REPORTS_DIR / f"training_history_{run_name}.csv"
    history = pd.read_csv(history_path)
    eval_rows = history.dropna(subset=["eval_f1"]) if "eval_f1" in history.columns else pd.DataFrame()
    if eval_rows.empty:
        return {"run_name": run_name, "status": "no_eval_history"}

    best_eval_f1 = float(eval_rows["eval_f1"].max())
    last_eval_f1 = float(eval_rows["eval_f1"].iloc[-1])
    best_epoch = float(eval_rows.loc[eval_rows["eval_f1"].idxmax(), "epoch"])
    last_eval_loss = float(eval_rows["eval_loss"].iloc[-1]) if "eval_loss" in eval_rows else math.nan
    best_eval_loss = float(eval_rows["eval_loss"].min()) if "eval_loss" in eval_rows else math.nan

    possible_underfit = best_eval_f1 < 0.60
    possible_overfit = (best_eval_f1 - last_eval_f1) > 0.03 or (not math.isnan(last_eval_loss) and last_eval_loss > best_eval_loss * 1.20)

    return {
        "run_name": run_name,
        "best_epoch": best_epoch,
        "best_eval_f1": best_eval_f1,
        "last_eval_f1": last_eval_f1,
        "possible_underfit": possible_underfit,
        "possible_overfit": possible_overfit,
    }


diagnostics_df = pd.DataFrame([diagnose_training(run_name) for run_name in MODEL_CONFIGS])
diagnostics_df.to_csv(REPORTS_DIR / "fit_diagnostics.csv", index=False)
display(diagnostics_df)

## 20. Pilih Model Terbaik

Model terbaik dipilih berdasarkan `test_f1`. Jika selisih F1 kecil, pertimbangkan stabilitas per entity, ukuran model, dan inference time untuk web ONNX.

In [ ]:
best_row = comparison_df.sort_values("test_f1", ascending=False).iloc[0]
best_run_name = best_row["run_name"]
best_model_dir = Path(best_row["best_model_dir"])

best_model_info = {
    "best_run_name": best_run_name,
    "best_model_dir": str(best_model_dir),
    "selection_metric": "test_f1",
    "test_f1": float(best_row["test_f1"]),
}
(OUTPUT_DIR / "best_model.json").write_text(json.dumps(best_model_info, indent=2), encoding="utf-8")

print(json.dumps(best_model_info, indent=2))

## 21. Error Analysis Model Terbaik

Error analysis membantu menjelaskan kelemahan model. Cell ini mengambil sampel kalimat test yang prediksinya berbeda dari label gold, lalu menyimpan tabel untuk laporan.

In [ ]:
def get_sentence_predictions(model_dir: Path, limit: int = 80) -> pd.DataFrame:
    tokenizer, tokenized_dataset = tokenize_and_align_dataset(raw_dataset, str(model_dir), MAX_LENGTH)
    model = AutoModelForTokenClassification.from_pretrained(model_dir)
    trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir=str(OUTPUT_DIR / "tmp_predict"), report_to="none", per_device_eval_batch_size=16),
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    )
    predictions, labels, _ = trainer.predict(tokenized_dataset["test"])
    pred_labels, gold_labels = decode_predictions(predictions, labels)

    rows = []
    for idx, (pred, gold, raw) in enumerate(zip(pred_labels, gold_labels, test_records)):
        if pred != gold:
            rows.append({
                "test_index": idx,
                "sentence": " ".join(raw["tokens"]),
                "gold_labels": " ".join(gold),
                "pred_labels": " ".join(pred),
            })
        if len(rows) >= limit:
            break
    return pd.DataFrame(rows)


error_df = get_sentence_predictions(best_model_dir, limit=80)
error_df.to_csv(REPORTS_DIR / "error_analysis_samples.csv", index=False)
display(error_df.head(20))

## 22. Export Model Terbaik ke ONNX FP32

ONNX FP32 digunakan sebagai baseline deployment. Pada tahap ini akurasi seharusnya tidak berubah signifikan karena bobot belum di-quantize.

In [ ]:
onnx_fp32_dir = ONNX_DIR / "fp32"
onnx_fp32_dir.mkdir(parents=True, exist_ok=True)

ort_model = ORTModelForTokenClassification.from_pretrained(best_model_dir, export=True)
best_tokenizer = AutoTokenizer.from_pretrained(best_model_dir)
ort_model.save_pretrained(onnx_fp32_dir)
best_tokenizer.save_pretrained(onnx_fp32_dir)

label_map = {"label_list": LABEL_LIST, "label2id": LABEL2ID, "id2label": ID2LABEL}
(onnx_fp32_dir / "label_map.json").write_text(json.dumps(label_map, indent=2), encoding="utf-8")

onnx_files = list(onnx_fp32_dir.glob("*.onnx"))
if not onnx_files:
    raise FileNotFoundError(f"Tidak ada file ONNX di {onnx_fp32_dir}")
onnx_fp32_path = onnx_files[0]
print("ONNX FP32:", onnx_fp32_path, f"{onnx_fp32_path.stat().st_size / (1024 ** 2):.2f} MB")

## 23. Quantization ONNX

Dynamic quantization dibuat untuk menurunkan ukuran model. Model quantized hanya dipakai untuk web jika hasil validasi tetap stabil.

In [ ]:
onnx_quantized_dir = ONNX_DIR / "quantized"
onnx_quantized_dir.mkdir(parents=True, exist_ok=True)
onnx_quantized_path = onnx_quantized_dir / "model_quantized.onnx"

quantize_dynamic(str(onnx_fp32_path), str(onnx_quantized_path), weight_type=QuantType.QInt8)

for file_name in ["tokenizer.json", "vocab.txt", "tokenizer_config.json", "special_tokens_map.json", "config.json", "label_map.json"]:
    source_file = onnx_fp32_dir / file_name
    if source_file.exists():
        shutil.copy2(source_file, onnx_quantized_dir / file_name)

size_report = {
    "fp32_mb": onnx_fp32_path.stat().st_size / (1024 ** 2),
    "quantized_mb": onnx_quantized_path.stat().st_size / (1024 ** 2),
}
(REPORTS_DIR / "onnx_size_report.json").write_text(json.dumps(size_report, indent=2), encoding="utf-8")
print(json.dumps(size_report, indent=2))

## 24. Siapkan Folder Model untuk Web

Cell ini membuat folder `web_model` yang nantinya disalin ke proyek lokal `web/model`. Default yang dipilih adalah ONNX quantized karena lebih ringan. Jika nanti validasi menunjukkan hasilnya turun terlalu jauh, gunakan ONNX FP32.

In [ ]:
WEB_MODEL_DIR.mkdir(parents=True, exist_ok=True)
preferred_onnx = onnx_quantized_path if onnx_quantized_path.exists() else onnx_fp32_path
shutil.copy2(preferred_onnx, WEB_MODEL_DIR / "model.onnx")

asset_source_dir = onnx_quantized_dir if preferred_onnx == onnx_quantized_path else onnx_fp32_dir
for file_name in ["tokenizer.json", "vocab.txt", "tokenizer_config.json", "special_tokens_map.json", "config.json", "label_map.json"]:
    source_file = asset_source_dir / file_name
    if source_file.exists():
        shutil.copy2(source_file, WEB_MODEL_DIR / file_name)

print("Web model folder:", WEB_MODEL_DIR)
print("Files:")
for path in sorted(WEB_MODEL_DIR.iterdir()):
    print("-", path.name, f"{path.stat().st_size / (1024 ** 2):.2f} MB")

## 25. Final Checklist Output

Cell terakhir memastikan file penting untuk laporan, slide, dan web demo sudah terbentuk. Download `/kaggle/working/outputs` setelah notebook selesai.

In [ ]:
expected_outputs = [
    OUTPUT_DIR / "dataset_stats.json",
    REPORTS_DIR / "model_comparison.csv",
    REPORTS_DIR / "fit_diagnostics.csv",
    REPORTS_DIR / "error_analysis_samples.csv",
    OUTPUT_DIR / "best_model.json",
    WEB_MODEL_DIR / "model.onnx",
    WEB_MODEL_DIR / "label_map.json",
]

checklist = pd.DataFrame({
    "path": [str(path) for path in expected_outputs],
    "exists": [path.exists() for path in expected_outputs],
})
display(checklist)

if not checklist["exists"].all():
    missing = checklist.loc[~checklist["exists"], "path"].tolist()
    raise FileNotFoundError(f"Output belum lengkap: {missing}")

print("Notebook selesai. Download /kaggle/working/outputs dan salin outputs/web_model ke web/model lokal.")